# Module 4 Lab 04 - File I/O and ETL Load Back to SQL Server

This notebook performs a complete mini-ETL: extract, clean, validate, export CSV/Excel, and load cleaned results back to SQL Server when a connection is available.

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pandas", "numpy", "sqlalchemy", "python-dotenv", "openpyxl"])

# pyodbc is needed only for the optional SQL Server load-back step.
# If installation fails in Colab, the file I/O and transformation sections still run.
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyodbc"])
except Exception as exc:
    print("pyodbc install skipped or failed. SQL Server load-back may be skipped:", exc)

import os
from pathlib import Path
from urllib.parse import quote_plus

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

try:
    import pyodbc
    PYODBC_AVAILABLE = True
except Exception as exc:
    PYODBC_AVAILABLE = False
    print("pyodbc is unavailable; SQL Server load-back will be skipped:", exc)

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
SAMPLE_CSV = DATA_DIR / "m4_raw_financial_transactions_sample.csv"

def load_sample_transactions():
    if SAMPLE_CSV.exists():
        return pd.read_csv(SAMPLE_CSV)
    print("Local CSV not found; using built-in fallback sample rows.")
    return pd.DataFrame([
        {"SourceSystem": "CoreBanking", "TransactionReference": "M4-0001", "TransactionDateText": "2026-06-01", "InstitutionCode": "MCB", "CounterpartyName": "Maseru Commercial Bank", "CurrencyCode": "LSL", "AmountText": "15000.00", "TransactionType": "Deposit", "Channel": "Branch", "LoadBatch": "BATCH-202606"},
        {"SourceSystem": "Payments", "TransactionReference": "M4-0003", "TransactionDateText": "2026/06/02", "InstitutionCode": "CPS", "CounterpartyName": "Cape Payment Services", "CurrencyCode": "ZAR", "AmountText": "25000", "TransactionType": "Transfer", "Channel": "Clearing", "LoadBatch": "BATCH-202606"},
        {"SourceSystem": "Treasury", "TransactionReference": "M4-0004", "TransactionDateText": "2026-06-03", "InstitutionCode": "CBL", "CounterpartyName": "Central Bank of Lesotho", "CurrencyCode": "USD", "AmountText": "100000.00", "TransactionType": "FX Settlement", "Channel": "SWIFT", "LoadBatch": "BATCH-202606"},
        {"SourceSystem": "CoreBanking", "TransactionReference": "M4-0006", "TransactionDateText": None, "InstitutionCode": "MCB", "CounterpartyName": "Maseru Commercial Bank", "CurrencyCode": "LSL", "AmountText": "9100.00", "TransactionType": "Deposit", "Channel": "Branch", "LoadBatch": "BATCH-202606"},
        {"SourceSystem": "Treasury", "TransactionReference": "M4-0010", "TransactionDateText": "2026-06-08", "InstitutionCode": "CBL", "CounterpartyName": "Central Bank of Lesotho", "CurrencyCode": "usd", "AmountText": "bad_amount", "TransactionType": "FX Settlement", "Channel": "SWIFT", "LoadBatch": "BATCH-202606"},
    ])

In [ ]:
# Connection helper. In Colab, set these environment variables manually if SQL Server is reachable.
load_dotenv(BASE_DIR / ".env")

def get_engine():
    if not PYODBC_AVAILABLE:
        raise RuntimeError("pyodbc is not installed")
    driver = os.getenv("DB_DRIVER", "ODBC Driver 18 for SQL Server")
    server = os.getenv("DB_SERVER", "localhost,1433")
    database = os.getenv("DB_NAME", "TrainingDB")
    user = os.getenv("DB_USER", "sa")
    password = os.getenv("DB_PASSWORD", "StrongPassw0rd!2026")
    trusted = os.getenv("DB_TRUSTED", "no").lower() in ("yes", "true", "1")
    parts = [
        f"DRIVER={{{driver}}}",
        f"SERVER={server}",
        f"DATABASE={database}",
        "Encrypt=yes",
        "TrustServerCertificate=yes",
    ]
    if trusted:
        parts.append("Trusted_Connection=yes")
    else:
        parts.extend([f"UID={user}", f"PWD={password}"])
    return create_engine(f"mssql+pyodbc:///?odbc_connect={quote_plus(';'.join(parts) + ';')}", fast_executemany=True)

In [ ]:
# Extract from SQL Server when possible; otherwise use local CSV.
sql = """
SELECT
    RawTransactionID,
    SourceSystem,
    TransactionReference,
    TransactionDateText,
    InstitutionCode,
    CounterpartyName,
    CurrencyCode,
    AmountText,
    TransactionType,
    Channel,
    LoadBatch
FROM m4.RawFinancialTransactions
ORDER BY RawTransactionID;
"""

engine = None
try:
    engine = get_engine()
    raw_df = pd.read_sql(sql, engine)
    source = "SQL Server"
except Exception as exc:
    print("Using CSV fallback. SQL Server not available:", exc)
    raw_df = load_sample_transactions()
    raw_df.insert(0, "RawTransactionID", range(1, len(raw_df) + 1))
    source = "CSV fallback"

print("Source:", source)
display(raw_df.head())

In [ ]:
# Transform and validate.
FX_RATES_TO_LSL = {"LSL": 1.0, "ZAR": 1.0, "USD": 18.25, "EUR": 19.75, "GBP": 23.10}

df = raw_df.copy()
df["TransactionDate"] = pd.to_datetime(df["TransactionDateText"], format="mixed", errors="coerce")
df["CurrencyCode"] = df["CurrencyCode"].astype("string").str.strip().str.upper()
df["Amount"] = pd.to_numeric(df["AmountText"], errors="coerce")
df["InstitutionCode"] = df["InstitutionCode"].astype("string").str.strip().str.upper()
df["CounterpartyName"] = df["CounterpartyName"].fillna("Unknown Counterparty")
df["TransactionType"] = df["TransactionType"].fillna("Unclassified")
df["Channel"] = df["Channel"].fillna("Unknown")

required = ["TransactionDate", "InstitutionCode", "CurrencyCode", "Amount", "TransactionType", "Channel"]
valid_mask = df[required].notna().all(axis=1) & df["CurrencyCode"].isin(FX_RATES_TO_LSL) & (df["Amount"] >= 0)

clean_df = df.loc[valid_mask].copy()
rejected_df = df.loc[~valid_mask].copy()
clean_df["TransactionDate"] = clean_df["TransactionDate"].dt.date
clean_df["FxRateToLSL"] = clean_df["CurrencyCode"].map(FX_RATES_TO_LSL)
clean_df["AmountLSL"] = (clean_df["Amount"] * clean_df["FxRateToLSL"]).round(2)
clean_df["AmountBand"] = pd.cut(clean_df["Amount"], bins=[-0.01, 10000, 50000, np.inf], labels=["Small", "Medium", "Large"]).astype(str)

output_columns = ["RawTransactionID", "TransactionReference", "TransactionDate", "InstitutionCode", "CounterpartyName", "CurrencyCode", "Amount", "AmountLSL", "TransactionType", "Channel", "AmountBand"]
clean_df = clean_df[output_columns]

# Three-step validation gate before any DataFrame is loaded back to SQL Server.
# 1) Null check for target non-nullable columns using df.isnull().sum().
# 2) Data type review against the expected target table shape using df.dtypes.
# 3) Row-count reconciliation against the expected extraction count.
target_non_nullable_columns = output_columns
target_schema_notes = {
    "RawTransactionID": "integer",
    "TransactionReference": "string",
    "TransactionDate": "date",
    "InstitutionCode": "string",
    "CounterpartyName": "string",
    "CurrencyCode": "3-character string",
    "Amount": "numeric",
    "AmountLSL": "numeric",
    "TransactionType": "string",
    "Channel": "string",
    "AmountBand": "string",
}

null_counts = clean_df[target_non_nullable_columns].isnull().sum()
dtype_report = clean_df.dtypes.astype(str)
expected_extraction_count = len(raw_df)
reconciled_row_count = len(clean_df) + len(rejected_df)

print("VALIDATION GATE 1 - Null counts for non-nullable columns")
print(null_counts)
print("\nVALIDATION GATE 2 - DataFrame dtypes compared with target schema notes")
for column, expected_type in target_schema_notes.items():
    print(f"{column}: actual={dtype_report[column]} | expected={expected_type}")
print("\nVALIDATION GATE 3 - Row-count reconciliation")
print(f"Extracted rows={expected_extraction_count}; clean rows={len(clean_df)}; rejected rows={len(rejected_df)}; clean+rejected={reconciled_row_count}")

if null_counts.sum() > 0:
    raise ValueError("Validation gate failed: non-nullable target columns contain null values")
if expected_extraction_count != reconciled_row_count:
    raise ValueError("Validation gate failed: row-count reconciliation failed")

summary_df = clean_df.groupby("CurrencyCode", as_index=False).agg(
    TransactionCount=("TransactionReference", "count"),
    TotalAmount=("Amount", "sum"),
    TotalAmountLSL=("AmountLSL", "sum"),
    AverageAmount=("Amount", "mean"),
    StandardDeviationAmount=("Amount", "std"),
).fillna({"StandardDeviationAmount": 0}).round(2)

if np.round(clean_df["Amount"].to_numpy().sum(), 2) != np.round(summary_df["TotalAmount"].to_numpy().sum(), 2):
    raise ValueError("Detail-to-summary reconciliation failed")

print("\nVALIDATION GATE PASSED - DataFrame is ready for file export and optional SQL load-back.")

display(clean_df)
display(summary_df)
display(rejected_df[["TransactionReference", "TransactionDateText", "InstitutionCode", "CurrencyCode", "AmountText"]])

In [ ]:
# File I/O: write CSV and Excel outputs.
clean_csv = OUTPUT_DIR / "clean_transactions_notebook.csv"
summary_csv = OUTPUT_DIR / "currency_summary_notebook.csv"
clean_excel = OUTPUT_DIR / "clean_transactions_notebook.xlsx"

clean_df.to_csv(clean_csv, index=False)
summary_df.to_csv(summary_csv, index=False)
clean_df.to_excel(clean_excel, index=False)

print("Wrote:")
print(clean_csv)
print(summary_csv)
print(clean_excel)

In [ ]:
# Load back to SQL Server if a connection is available.
# In Colab without SQL Server, this cell safely skips the database load.
if engine is None:
    print("SQL Server engine unavailable; skipping database load.")
else:
    with engine.begin() as conn:
        run_id = conn.execute(text("""
            INSERT INTO m4.EtlRunLog
                (Status, ExtractedRows, CleanRows, RejectedRows, Message)
            OUTPUT inserted.RunID
            VALUES
                ('Running', :extracted_rows, :clean_rows, :rejected_rows, 'Notebook ETL started');
        """), {
            "extracted_rows": len(raw_df),
            "clean_rows": len(clean_df),
            "rejected_rows": len(rejected_df),
        }).scalar_one()

        conn.execute(text("DELETE FROM m4.CleanFinancialTransactions;"))
        conn.execute(text("DELETE FROM m4.CurrencySummary;"))
        clean_df.to_sql("CleanFinancialTransactions", con=conn, schema="m4", if_exists="append", index=False)
        summary_df.to_sql("CurrencySummary", con=conn, schema="m4", if_exists="append", index=False)

        loaded_clean_rows = conn.execute(text("SELECT COUNT(*) FROM m4.CleanFinancialTransactions;")).scalar_one()
        loaded_summary_rows = conn.execute(text("SELECT COUNT(*) FROM m4.CurrencySummary;")).scalar_one()
        if loaded_clean_rows != len(clean_df):
            raise ValueError(f"SQL load verification failed: expected {len(clean_df)} clean rows, found {loaded_clean_rows}")
        if loaded_summary_rows != len(summary_df):
            raise ValueError(f"SQL load verification failed: expected {len(summary_df)} summary rows, found {loaded_summary_rows}")

        conn.execute(text("""
            UPDATE m4.EtlRunLog
            SET CompletedAt = SYSUTCDATETIME(), Status = 'Succeeded', Message = 'Notebook ETL completed successfully. Null, dtype, row-count, and SQL load checks passed.'
            WHERE RunID = :run_id;
        """), {"run_id": run_id})
    print("Loaded notebook ETL results to SQL Server. Run ID:", run_id)
    print("SQL verification clean rows:", loaded_clean_rows)
    print("SQL verification summary rows:", loaded_summary_rows)